# LeetCode 1306 — Jump Game III bằng Python

Link đề: https://leetcode.com/problems/jump-game-iii/ · Độ khó: **Medium** · Chủ đề: Array, Depth-First Search, Breadth-First Search

Skeleton LeetCode:

```python
class Solution:
    def canReach(self, arr: list[int], start: int) -> bool:
        ...
```

## 1. Mục tiêu học

1. Biết **mô hình hoá mảng thành đồ thị**: mỗi chỉ số là một đỉnh, mỗi bước nhảy hợp lệ là một cạnh có hướng.
2. Giải bài **"có đến được không?" (reachability)** bằng **DFS** (đệ quy và lặp) và **BFS** với tập `visited`.
3. Hiểu vì sao **bắt buộc** phải đánh dấu đã thăm (đồ thị có chu trình) và cách **đánh dấu tại chỗ** để tiết kiệm bộ nhớ.
4. Kiểm soát biên mảng và giới hạn đệ quy của Python với `n` tới `5·10^4`.
5. Tránh các lỗi: quên `visited`, sai điều kiện biên, đánh dấu tại chỗ bằng giá trị gây "dương tính giả", quên kiểm tra chính vị trí `start`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên **không âm** `arr` và chỉ số bắt đầu `start`. Khi đang ở chỉ số `i`, bạn có thể nhảy tới `i + arr[i]` **hoặc** `i − arr[i]`, nhưng **không được nhảy ra ngoài mảng**. Hỏi có thể tới được **một chỉ số bất kỳ có giá trị 0** hay không?

**Ví dụ 1:** `arr = [4,2,3,0,3,1,2], start = 5` → `true` (ví dụ đường đi `5 → 4 → 1 → 3`).

**Ví dụ 2:** `arr = [4,2,3,0,3,1,2], start = 0` → `true` (`0 → 4 → 1 → 3`).

**Ví dụ 3:** `arr = [3,0,2,1,2], start = 2` → `false` (không cách nào tới chỉ số 1).

**Ràng buộc:** `1 <= arr.length <= 5·10^4`, `0 <= arr[i] < arr.length`, `0 <= start < arr.length`.

In [1]:
import random
import sys
import time
from collections import deque
from typing import List

examples = [
    (([4, 2, 3, 0, 3, 1, 2], 5), True),
    (([4, 2, 3, 0, 3, 1, 2], 0), True),
    (([3, 0, 2, 1, 2], 2), False),
]
for (arr, start), want in examples:
    print(f"arr={arr}, start={start} -> mong đợi {want}")

arr=[4, 2, 3, 0, 3, 1, 2], start=5 -> mong đợi True
arr=[4, 2, 3, 0, 3, 1, 2], start=0 -> mong đợi True
arr=[3, 0, 2, 1, 2], start=2 -> mong đợi False


## 3. Kiến thức nền

### 3.1 Mảng như một đồ thị

Đỉnh: chỉ số `0..n−1`. Từ đỉnh `i` có tối đa 2 cạnh: tới `i + arr[i]` và `i − arr[i]` (nếu nằm trong `[0, n−1]`). Đồ thị **có hướng** và có thể có **chu trình** (ví dụ `i → j → i`). Bài toán = "từ `start` có đi tới một đỉnh mang giá trị 0 không?".

In [2]:
def neighbors(arr, i):
    n = len(arr)
    res = []
    for j in (i + arr[i], i - arr[i]):
        if 0 <= j < n:
            res.append(j)
    return res

arr = [4, 2, 3, 0, 3, 1, 2]
for i in range(len(arr)):
    print(f"đỉnh {i} (arr={arr[i]}) -> {neighbors(arr, i)}")

đỉnh 0 (arr=4) -> [4]
đỉnh 1 (arr=2) -> [3]
đỉnh 2 (arr=3) -> [5]
đỉnh 3 (arr=0) -> [3, 3]
đỉnh 4 (arr=3) -> [1]
đỉnh 5 (arr=1) -> [6, 4]
đỉnh 6 (arr=2) -> [4]


### 3.2 Tập `set` để đánh dấu đã thăm

`x in s` và `s.add(x)` đều `O(1)` trung bình. Có thể thay bằng list boolean `seen = [False] * n` (nhanh và gọn khi đỉnh là số nguyên `0..n−1`).

In [3]:
seen = set()
seen.add(3); seen.add(5)
print(3 in seen, 4 in seen)
seen_list = [False] * 7
seen_list[3] = True
print(seen_list)

True False
[False, False, False, True, False, False, False]


### 3.3 DFS đệ quy và giới hạn đệ quy

DFS: thăm đỉnh, rồi đệ quy vào từng hàng xóm chưa thăm. Độ sâu đệ quy có thể tới `n = 5·10^4`, vượt giới hạn mặc định của Python (~1000) ⇒ cần `sys.setrecursionlimit` hoặc viết bằng stack.

In [4]:
def dfs_order(arr, start):
    order, seen = [], set()
    def dfs(i):
        seen.add(i)
        order.append(i)
        for j in neighbors(arr, i):
            if j not in seen:
                dfs(j)
    dfs(start)
    return order

print("thứ tự DFS từ 5:", dfs_order([4, 2, 3, 0, 3, 1, 2], 5))
print("giới hạn đệ quy:", sys.getrecursionlimit())

thứ tự DFS từ 5: [5, 6, 4, 1, 3]
giới hạn đệ quy: 1000


### 3.4 BFS với `deque`

BFS thăm theo "lớp khoảng cách": các đỉnh cách `start` 1 bước, rồi 2 bước, … `deque.popleft()` là `O(1)`. Quy tắc vàng: **đánh dấu khi đẩy vào hàng đợi** (không phải khi lấy ra) để mỗi đỉnh vào hàng đợi đúng một lần.

In [5]:
def bfs_order(arr, start):
    q = deque([start])
    seen = {start}
    order = []
    while q:
        i = q.popleft()
        order.append(i)
        for j in neighbors(arr, i):
            if j not in seen:
                seen.add(j)
                q.append(j)
    return order

print("thứ tự BFS từ 5:", bfs_order([4, 2, 3, 0, 3, 1, 2], 5))

thứ tự BFS từ 5: [5, 6, 4, 1, 3]


### 3.5 Stack tường minh (DFS lặp)

Thay đệ quy bằng list làm stack: `append` để đẩy, `pop()` để lấy phần tử cuối. Không bị giới hạn đệ quy.

In [6]:
stack = [5]
seen = {5}
order = []
arr = [4, 2, 3, 0, 3, 1, 2]
while stack:
    i = stack.pop()
    order.append(i)
    for j in neighbors(arr, i):
        if j not in seen:
            seen.add(j)
            stack.append(j)
print("DFS lặp từ 5:", order)

DFS lặp từ 5: [5, 4, 1, 3, 6]


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 3**: `arr = [3,0,2,1,2]`, `start = 2`. Số 0 duy nhất ở chỉ số **1**.

| đỉnh | arr | nhảy +  | nhảy − | hàng xóm hợp lệ |
|---|---|---|---|---|
| 0 | 3 | 3 | −3 ✘ | [3] |
| 1 | 0 | 1 | 1 | [1] (đích) |
| 2 | 2 | 4 | 0 | [4, 0] |
| 3 | 1 | 4 | 2 | [4, 2] |
| 4 | 2 | 6 ✘ | 2 | [2] |

Từ `2` đi được `{4, 0}`; từ `0` đi `3`; từ `3` đi `{4, 2}`; từ `4` về `2`. Tập đến được = `{0, 2, 3, 4}` — **không chứa 1** ⇒ `false`.

Quan sát: có chu trình `2 → 4 → 2` ⇒ nếu không ghi nhớ đỉnh đã thăm, ta sẽ đi vòng mãi. Mỗi đỉnh chỉ cần thăm **một lần**: lần thứ hai không mở ra điều gì mới.

In [7]:
arr, start = [3, 0, 2, 1, 2], 2
reach = bfs_order(arr, start)
print("tập đến được:", sorted(reach), "| có chỉ số mang 0?", any(arr[i] == 0 for i in reach))

tập đến được: [0, 2, 3, 4] | có chỉ số mang 0? False


## 5. Cách 1 — Brute Force / baseline: lan truyền lặp đến khi ổn định

**Ý tưởng:** giữ mảng `reach[i]` (đến được chưa). Ban đầu chỉ `reach[start] = True`. Lặp: quét **toàn bộ** mảng, với mỗi `i` đã đến được thì đánh dấu hai đích nhảy của nó. Khi một lượt quét không thay đổi gì ⇒ dừng. Cuối cùng kiểm tra có `i` nào `reach[i]` và `arr[i] == 0`.

Đây là cách "không cần biết DFS/BFS" — giống thuật toán Bellman–Ford cho tính đến được.

In [8]:
def can_reach_brute(arr, start):
    n = len(arr)
    reach = [False] * n
    reach[start] = True
    changed = True
    rounds = 0
    while changed:
        changed = False
        rounds += 1
        for i in range(n):
            if reach[i]:
                for j in (i + arr[i], i - arr[i]):
                    if 0 <= j < n and not reach[j]:
                        reach[j] = True
                        changed = True
    return any(reach[i] and arr[i] == 0 for i in range(n))

for (arr, start), want in examples:
    print(arr, start, "->", can_reach_brute(arr, start), "| mong đợi", want)

[4, 2, 3, 0, 3, 1, 2] 5 -> True | mong đợi True
[4, 2, 3, 0, 3, 1, 2] 0 -> True | mong đợi True
[3, 0, 2, 1, 2] 2 -> False | mong đợi False


**Độ phức tạp:** mỗi lượt quét `O(n)`; mỗi lượt (trừ lượt cuối) thêm ít nhất 1 đỉnh ⇒ tối đa `n` lượt ⇒ Time `O(n²)` trường hợp xấu (`2.5·10^9` với `n = 5·10^4` — quá chậm). Space `O(n)`.

**Nút thắt:** mỗi lượt quét lại cả những đỉnh đã xử lý xong. Ta nên chỉ xử lý **đỉnh vừa mới đến được** — đó chính là DFS/BFS.

In [9]:
# Trường hợp xấu cho brute force: chuỗi nhảy "ngược chiều quét"
def worst_case(n):
    # từ chỉ số n-1 nhảy lùi từng bước 1 về 0, số 0 nằm ở đầu
    return [0] + [1] * (n - 1), n - 1

for n in (500, 1000, 2000):
    arr, s = worst_case(n)
    t0 = time.perf_counter()
    r = can_reach_brute(arr, s)
    print(f"n={n}: {r} ({time.perf_counter() - t0:.3f}s) — tăng ~4 lần khi n gấp đôi")

n=500: True (0.024s) — tăng ~4 lần khi n gấp đôi
n=1000: True (0.102s) — tăng ~4 lần khi n gấp đôi


n=2000: True (0.409s) — tăng ~4 lần khi n gấp đôi


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** "Có đến được đỉnh đích không?" là bài **duyệt đồ thị** kinh điển. Chỉ cần thăm mỗi đỉnh **đúng một lần** và đi theo các cạnh của nó:
- **Nếu** đỉnh hiện tại có `arr[i] == 0` **thì** trả `True` ngay.
- **Nếu** đỉnh đã thăm **thì** bỏ qua (lần thăm trước đã khám phá mọi thứ từ nó).
- **Nếu** nhảy ra ngoài `[0, n−1]` **thì** bỏ qua.
- Duyệt hết mà không gặp 0 ⇒ `False`.

**Invariant:** tập `visited` luôn là tập con của các đỉnh đến được từ `start`; khi duyệt kết thúc, `visited` **bằng đúng** tập đến được.

Vì mỗi đỉnh có tối đa 2 cạnh, tổng công việc là `O(n)`.

**Mẹo tiết kiệm bộ nhớ:** vì `arr[i] >= 0`, ta có thể đánh dấu đã thăm bằng cách **đổi dấu** `arr[i] = -arr[i]` (giá trị âm = đã thăm). Riêng `arr[i] == 0` là đích — trả `True` trước khi cần đánh dấu.

In [10]:
# Minh hoạ: mỗi đỉnh được xử lý đúng 1 lần dù có chu trình
arr, start = [3, 0, 2, 1, 2], 2
seen, stack, visits = {start}, [start], []
while stack:
    i = stack.pop()
    visits.append(i)
    for j in neighbors(arr, i):
        if j not in seen:
            seen.add(j); stack.append(j)
print("các đỉnh được xử lý:", visits, "-> không đỉnh nào lặp lại")

các đỉnh được xử lý: [2, 0, 3, 4] -> không đỉnh nào lặp lại


## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS đệ quy với `visited`

Hàm `dfs(i)` trả `True` nếu từ `i` đến được một số 0. Ba kiểm tra đầu hàm: ngoài biên, đã thăm, là đích.

In [11]:
def can_reach_dfs(arr, start):
    n = len(arr)
    seen = [False] * n
    def dfs(i):
        if i < 0 or i >= n or seen[i]:
            return False
        if arr[i] == 0:
            return True
        seen[i] = True
        return dfs(i + arr[i]) or dfs(i - arr[i])   # "or" dừng sớm khi nhánh trái đúng
    return dfs(start)

for (arr, start), want in examples:
    print(arr, start, "->", can_reach_dfs(arr, start), "| mong đợi", want)

[4, 2, 3, 0, 3, 1, 2] 5 -> True | mong đợi True
[4, 2, 3, 0, 3, 1, 2] 0 -> True | mong đợi True
[3, 0, 2, 1, 2] 2 -> False | mong đợi False


Code nộp LeetCode (cách 2, nâng giới hạn đệ quy vì độ sâu có thể tới `5·10^4`):

In [12]:
class SolutionDFS:
    def canReach(self, arr: list[int], start: int) -> bool:
        sys.setrecursionlimit(max(sys.getrecursionlimit(), 100000))
        n = len(arr)
        seen = [False] * n
        def dfs(i: int) -> bool:
            if i < 0 or i >= n or seen[i]:
                return False
            if arr[i] == 0:
                return True
            seen[i] = True
            return dfs(i + arr[i]) or dfs(i - arr[i])
        return dfs(start)

print(SolutionDFS().canReach([4, 2, 3, 0, 3, 1, 2], 5))

True


**Độ phức tạp cách 2:** Time `O(n)`, Space `O(n)` (mảng `seen` + stack đệ quy sâu tới `n`).

### 7.2 Cách 3 — BFS với hàng đợi

Không đệ quy, an toàn tuyệt đối với `n` lớn. Kiểm tra đích ngay khi lấy đỉnh ra (hoặc khi đẩy vào). BFS còn cho thêm thông tin: **số bước nhảy ít nhất** (xem bài tập 5).

In [13]:
def can_reach_bfs(arr, start):
    n = len(arr)
    seen = [False] * n
    seen[start] = True
    q = deque([start])
    while q:
        i = q.popleft()
        if arr[i] == 0:
            return True
        for j in (i + arr[i], i - arr[i]):
            if 0 <= j < n and not seen[j]:
                seen[j] = True                  # đánh dấu khi đẩy vào
                q.append(j)
    return False

for (arr, start), want in examples:
    print(arr, start, "->", can_reach_bfs(arr, start), "| mong đợi", want)

[4, 2, 3, 0, 3, 1, 2] 5 -> True | mong đợi True
[4, 2, 3, 0, 3, 1, 2] 0 -> True | mong đợi True
[3, 0, 2, 1, 2] 2 -> False | mong đợi False


Code nộp LeetCode (cách 3):

In [14]:
class SolutionBFS:
    def canReach(self, arr: list[int], start: int) -> bool:
        n = len(arr)
        seen = [False] * n
        seen[start] = True
        q = deque([start])
        while q:
            i = q.popleft()
            if arr[i] == 0:
                return True
            for j in (i + arr[i], i - arr[i]):
                if 0 <= j < n and not seen[j]:
                    seen[j] = True
                    q.append(j)
        return False

print(SolutionBFS().canReach([3, 0, 2, 1, 2], 2))

False


**Độ phức tạp cách 3:** Time `O(n)`, Space `O(n)` (mảng `seen` + hàng đợi).

### 7.3 Cách 4 — DFS lặp, đánh dấu tại chỗ bằng đổi dấu

Không cần mảng `seen`: khi thăm `i` (với `arr[i] > 0`), gán `arr[i] = -arr[i]`. Một đỉnh có `arr[i] < 0` là đã thăm. Để tính bước nhảy cần độ dài gốc ⇒ lưu `step = arr[i]` **trước** khi đổi dấu.

Lưu ý: cách này **sửa input**; trong phỏng vấn nên hỏi trước, hoặc khôi phục lại (đổi dấu lại các phần tử âm) trước khi trả về — bản dưới làm đúng như vậy.

In [15]:
def can_reach_inplace(arr, start):
    n = len(arr)
    stack = [start]
    found = False
    while stack:
        i = stack.pop()
        if arr[i] == 0:
            found = True
            break
        if arr[i] < 0:                      # đã thăm
            continue
        step = arr[i]
        arr[i] = -step                      # đánh dấu đã thăm
        for j in (i + step, i - step):
            if 0 <= j < n and arr[j] >= 0:
                stack.append(j)
    for k in range(n):                      # khôi phục input
        if arr[k] < 0:
            arr[k] = -arr[k]
    return found

for (arr, start), want in examples:
    a = list(arr)
    print(arr, start, "->", can_reach_inplace(a, start), "| mong đợi", want, "| input được khôi phục:", a == arr)

[4, 2, 3, 0, 3, 1, 2] 5 -> True | mong đợi True | input được khôi phục: True
[4, 2, 3, 0, 3, 1, 2] 0 -> True | mong đợi True | input được khôi phục: True
[3, 0, 2, 1, 2] 2 -> False | mong đợi False | input được khôi phục: True


Code nộp LeetCode (cách 4 — bản gọn, chấp nhận sửa input):

In [16]:
class SolutionInPlace:
    def canReach(self, arr: list[int], start: int) -> bool:
        n = len(arr)
        stack = [start]
        while stack:
            i = stack.pop()
            if arr[i] == 0:
                return True
            if arr[i] < 0:
                continue
            step = arr[i]
            arr[i] = -step
            for j in (i + step, i - step):
                if 0 <= j < n and arr[j] >= 0:
                    stack.append(j)
        return False

print(SolutionInPlace().canReach([4, 2, 3, 0, 3, 1, 2], 0))

True


**Độ phức tạp cách 4:** Time `O(n)`, Space phụ `O(n)` cho stack trong trường hợp xấu nhưng không cần mảng `seen`. Một đỉnh có thể bị đẩy vào stack nhiều lần (kiểm tra `arr[j] >= 0` lúc đẩy chưa chặn hết trùng lặp), song mỗi lần đẩy ứng với một cạnh được xét từ một đỉnh vừa thăm, nên tổng số lần đẩy ≤ số cạnh ≤ `2n` ⇒ vẫn `O(n)`.

## 8. TRACE từng bước

Trace BFS (cách 3) trên Ví dụ 1: `arr = [4,2,3,0,3,1,2]`, `start = 5`.

| Bước | Lấy ra `i` | arr[i] | i+arr[i] | i−arr[i] | Đẩy vào | Hàng đợi sau |
|---|---|---|---|---|---|---|
| 1 | 5 | 1 | 6 | 4 | 6, 4 | [6, 4] |
| 2 | 6 | 2 | 8 ✘ | 4 (đã đánh dấu) | – | [4] |
| 3 | 4 | 3 | 7 ✘ | 1 | 1 | [1] |
| 4 | 1 | 2 | 3 | −1 ✘ | 3 | [3] |
| 5 | 3 | **0** | | | | ⇒ **True** |

In [17]:
def trace_bfs(arr, start):
    n = len(arr)
    seen = [False] * n
    seen[start] = True
    q = deque([start])
    step = 0
    while q:
        step += 1
        i = q.popleft()
        if arr[i] == 0:
            print(f"bước {step}: lấy {i}, arr[{i}] = 0 => True")
            return True
        pushed, notes = [], []
        for j in (i + arr[i], i - arr[i]):
            if not (0 <= j < n):
                notes.append(f"{j} ngoài biên")
            elif seen[j]:
                notes.append(f"{j} đã thăm")
            else:
                seen[j] = True
                q.append(j)
                pushed.append(j)
        print(f"bước {step}: lấy {i} (arr={arr[i]}) đẩy {pushed} {notes} -> hàng đợi {list(q)}")
    print("hàng đợi rỗng => False")
    return False

trace_bfs([4, 2, 3, 0, 3, 1, 2], 5)

bước 1: lấy 5 (arr=1) đẩy [6, 4] [] -> hàng đợi [6, 4]
bước 2: lấy 6 (arr=2) đẩy [] ['8 ngoài biên', '4 đã thăm'] -> hàng đợi [4]
bước 3: lấy 4 (arr=3) đẩy [1] ['7 ngoài biên'] -> hàng đợi [1]
bước 4: lấy 1 (arr=2) đẩy [3] ['-1 ngoài biên'] -> hàng đợi [3]
bước 5: lấy 3, arr[3] = 0 => True


True

In [18]:
trace_bfs([3, 0, 2, 1, 2], 2)

bước 1: lấy 2 (arr=2) đẩy [4, 0] [] -> hàng đợi [4, 0]
bước 2: lấy 4 (arr=2) đẩy [] ['6 ngoài biên', '2 đã thăm'] -> hàng đợi [0]
bước 3: lấy 0 (arr=3) đẩy [3] ['-3 ngoài biên'] -> hàng đợi [3]
bước 4: lấy 3 (arr=1) đẩy [] ['4 đã thăm', '2 đã thăm'] -> hàng đợi []
hàng đợi rỗng => False


False

Trace DFS đệ quy (cách 2) trên Ví dụ 2 — thụt lề theo độ sâu đệ quy:

In [19]:
def trace_dfs(arr, start):
    n = len(arr)
    seen = [False] * n
    def dfs(i, depth):
        pad = "  " * depth
        if i < 0 or i >= n:
            print(f"{pad}{i}: ngoài biên"); return False
        if seen[i]:
            print(f"{pad}{i}: đã thăm"); return False
        if arr[i] == 0:
            print(f"{pad}{i}: arr=0 -> TÌM THẤY"); return True
        seen[i] = True
        print(f"{pad}{i}: arr={arr[i]} -> thử {i + arr[i]} rồi {i - arr[i]}")
        return dfs(i + arr[i], depth + 1) or dfs(i - arr[i], depth + 1)
    res = dfs(start, 0)
    print("=>", res)
    return res

trace_dfs([4, 2, 3, 0, 3, 1, 2], 0)

0: arr=4 -> thử 4 rồi -4
  4: arr=3 -> thử 7 rồi 1
    7: ngoài biên
    1: arr=2 -> thử 3 rồi -1
      3: arr=0 -> TÌM THẤY
=> True


True

## 9. Code nộp LeetCode

Bản chính thức: **BFS** với mảng `seen` — không đệ quy, không sửa input, `O(n)`.

In [20]:
class Solution:
    def canReach(self, arr: list[int], start: int) -> bool:
        n = len(arr)
        seen = [False] * n
        seen[start] = True
        q = deque([start])
        while q:
            i = q.popleft()
            if arr[i] == 0:                    # tới được một số 0
                return True
            for j in (i + arr[i], i - arr[i]):
                if 0 <= j < n and not seen[j]: # trong biên và chưa thăm
                    seen[j] = True
                    q.append(j)
        return False                           # đã thăm hết tập đến được

sol = Solution()
for (arr, start), want in examples:
    got = sol.canReach(list(arr), start)
    print(arr, start, "->", got, "✔" if got == want else "✘")

[4, 2, 3, 0, 3, 1, 2] 5 -> True ✔
[4, 2, 3, 0, 3, 1, 2] 0 -> True ✔
[3, 0, 2, 1, 2] 2 -> False ✔


## 10. Vì sao thuật toán đúng

Gọi `R` là tập chỉ số đến được từ `start`.

1. **Mọi đỉnh được đẩy vào hàng đợi đều thuộc `R`:** `start ∈ R`; một đỉnh `j` chỉ được đẩy khi nó là đích nhảy hợp lệ của một đỉnh `i` đã thuộc `R` ⇒ `j ∈ R`.
2. **Mọi đỉnh của `R` đều được đẩy vào (nếu không dừng sớm):** giả sử `v ∈ R` không bao giờ được đẩy. Lấy đường đi `start = u0 → u1 → … → uk = v`; gọi `u_t` là đỉnh đầu tiên trên đường không được đẩy (`t >= 1`). Khi đó `u_{t−1}` đã được đẩy rồi lấy ra, và lúc xử lý nó ta đã xét cạnh `u_{t−1} → u_t` ⇒ `u_t` được đẩy (hoặc đã được đẩy trước đó). Mâu thuẫn.
3. Vậy thuật toán xem xét đúng tập `R`; trả `True` khi và chỉ khi `R` chứa một chỉ số có giá trị 0.

**Dừng:** mỗi đỉnh được đẩy tối đa một lần (đánh dấu khi đẩy) ⇒ vòng lặp chạy tối đa `n` lần — không bị kẹt trong chu trình.

## 11. Phân tích Time / Space

Gọi `n = len(arr)` (≤ `5·10^4`). Đồ thị có `n` đỉnh và tối đa `2n` cạnh.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Lan truyền lặp | `O(n²)` worst | `O(n)` | quét lại nhiều lần |
| 2. DFS đệ quy | `O(n)` | `O(n)` (seen + đệ quy) | cần nâng recursion limit |
| 3. BFS | `O(n)` | `O(n)` | bản nộp chính |
| 4. DFS lặp, đánh dấu tại chỗ | `O(n)` | `O(n)` stack worst, không cần `seen` | sửa input (có thể khôi phục) |

Trường hợp tốt nhất: `arr[start] == 0` ⇒ `O(1)` sau khởi tạo (riêng `seen = [False]*n` vẫn tốn `O(n)`; cách 4 thật sự `O(1)`).

## 12. Bộ test

Ví dụ đề, edge case (`n = 1`, `start` đã là 0, mọi phần tử 0, không có số 0, chu trình, nhảy ra biên), trường hợp lớn `n = 5·10^4`, và đối chiếu mọi cách với brute force trên input ngẫu nhiên đúng ràng buộc (`0 <= arr[i] < n`).

In [21]:
def fresh(f):
    return lambda arr, s: f(list(arr), s)      # truyền bản sao vì có cách sửa input

ALL = [fresh(f) for f in [can_reach_brute, can_reach_dfs, can_reach_bfs, can_reach_inplace,
                          Solution().canReach, SolutionDFS().canReach, SolutionBFS().canReach,
                          SolutionInPlace().canReach]]
tests = [
    (([4, 2, 3, 0, 3, 1, 2], 5), True),
    (([4, 2, 3, 0, 3, 1, 2], 0), True),
    (([3, 0, 2, 1, 2], 2), False),
    (([0], 0), True),                  # n = 1, start là 0
    (([1, 0], 1), True),               # start đã là 0
    (([1, 1, 1], 1), False),           # không có số 0
    (([0, 0, 0], 2), True),
    (([2, 1, 2, 0], 0), False),        # 0 -> 2 -> 0 chu trình, không tới 3
    (([1, 2, 0], 0), False),           # 0 -> 1 -> (3 ngoài biên / -1 ngoài biên)
]
for (arr, s), want in tests:
    for f in ALL:
        got = f(arr, s)
        assert got == want, (arr, s, got, want)
print("Tất cả", len(tests), "test cố định qua cho", len(ALL), "hàm")

Tất cả 9 test cố định qua cho 8 hàm


In [22]:
rng = random.Random(1306)
cnt_true = 0
for _ in range(2000):
    n = rng.randint(1, 25)
    arr = [rng.randint(0, n - 1) if rng.random() > 0.12 else 0 for _ in range(n)]
    s = rng.randrange(n)
    want = can_reach_brute(arr, s)
    cnt_true += want
    for f in ALL[1:]:
        assert f(arr, s) == want, (arr, s)
print("Đối chiếu 2000 input ngẫu nhiên: khớp | số ca True:", cnt_true)

Đối chiếu 2000 input ngẫu nhiên: khớp | số ca True: 1147


In [23]:
# Kích thước tối đa: chuỗi nhảy dài 5*10^4 (độ sâu đệ quy lớn) và mảng ngẫu nhiên
n = 5 * 10**4
arr_chain, s = worst_case(n)
t0 = time.perf_counter()
print("chuỗi dài: BFS", Solution().canReach(list(arr_chain), s),
      "| DFS đệ quy", SolutionDFS().canReach(list(arr_chain), s),
      "| in-place", SolutionInPlace().canReach(list(arr_chain), s),
      f"({time.perf_counter() - t0:.3f}s)")
rng = random.Random(9)
arr_r = [rng.randint(1, n - 1) for _ in range(n)]
print("ngẫu nhiên không có 0:", Solution().canReach(arr_r, 0))

chuỗi dài: BFS True | DFS đệ quy True | in-place True (0.044s)


ngẫu nhiên không có 0: False


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không đánh dấu đã thăm ⇒ lặp vô hạn trong chu trình

In [24]:
def wrong_no_visited(arr, start):
    def dfs(i):
        if i < 0 or i >= len(arr):
            return False
        if arr[i] == 0:
            return True
        return dfs(i + arr[i]) or dfs(i - arr[i])     # SAI: không nhớ đỉnh đã thăm
    return dfs(start)

old = sys.getrecursionlimit()
sys.setrecursionlimit(2000)
try:
    print(wrong_no_visited([3, 0, 2, 1, 2], 2))
except RecursionError:
    print("RecursionError: kẹt trong chu trình 2 -> 4 -> 2 -> ...")
finally:
    sys.setrecursionlimit(old)
print("đúng:", Solution().canReach([3, 0, 2, 1, 2], 2))

RecursionError: kẹt trong chu trình 2 -> 4 -> 2 -> ...
đúng: False


**Vì sao sai:** từ `2` nhảy `+2` tới `4`, từ `4` nhảy `−2` về `2`, cứ thế mãi. Trường hợp có đáp án `True` có thể "may mắn" đúng nếu nhánh đầu tiên gặp 0, nhưng ca `False` luôn treo. **Sửa:** mảng `seen`/`set` hoặc đánh dấu tại chỗ.

### Lỗi 2 — Sai điều kiện biên (`<=` thay vì `<`)

In [25]:
def wrong_bound(arr, start):
    n = len(arr)
    seen = [False] * n
    q = deque([start]); seen[start] = True
    while q:
        i = q.popleft()
        if arr[i] == 0:
            return True
        for j in (i + arr[i], i - arr[i]):
            if 0 <= j <= n and not seen[j]:          # SAI: j == n vượt biên
                seen[j] = True
                q.append(j)
    return False

try:
    print(wrong_bound([1, 2, 1], 1))
except IndexError as e:
    print("Bắt được IndexError:", e)
print("đúng:", Solution().canReach([1, 2, 1], 1))

Bắt được IndexError: list index out of range
đúng: False


**Vì sao sai:** chỉ số hợp lệ là `0..n−1`. Với `[1,2,1]`, từ `1` nhảy `+2` tới `3 == n` ⇒ `seen[3]` lỗi. Trong Python, chỉ số **âm** còn nguy hiểm hơn vì **không báo lỗi** (`arr[-1]` là phần tử cuối) ⇒ luôn kiểm tra cả `j >= 0`. **Sửa:** `0 <= j < n`.

### Lỗi 3 — Đánh dấu tại chỗ bằng cách gán `arr[i] = 0`

In [26]:
def wrong_mark_zero(arr, start):
    arr = list(arr)
    n = len(arr)
    stack = [start]
    while stack:
        i = stack.pop()
        if arr[i] == 0:
            return True
        step = arr[i]
        arr[i] = 0                                   # SAI: 0 là giá trị "đích"!
        for j in (i + step, i - step):
            if 0 <= j < n:
                stack.append(j)
    return False

print("sai :", wrong_mark_zero([3, 0, 2, 1, 2], 2))
print("đúng:", Solution().canReach([3, 0, 2, 1, 2], 2))

sai : True
đúng: False


**Vì sao sai:** sau khi thăm `2` ta gán `arr[2] = 0`; khi đi `2 → 4 → 2` quay lại, thuật toán thấy `arr[2] == 0` và tưởng đã tới đích ⇒ **dương tính giả**. **Sửa:** dùng một dấu hiệu **không thể nhầm** với dữ liệu thật: đổi dấu thành âm (vì `arr[i] >= 0`), hoặc mảng `seen` riêng.

### Lỗi 4 — Chỉ kiểm tra số 0 ở các đỉnh hàng xóm, quên chính `start`

In [27]:
def wrong_skip_start(arr, start):
    n = len(arr)
    seen = {start}
    q = deque([start])
    while q:
        i = q.popleft()
        for j in (i + arr[i], i - arr[i]):
            if 0 <= j < n and j not in seen:
                if arr[j] == 0:                      # chỉ xét hàng xóm
                    return True
                seen.add(j)
                q.append(j)
    return False                                     # SAI khi arr[start] == 0

print("sai :", wrong_skip_start([0, 1], 0), "| đúng:", Solution().canReach([0, 1], 0))
print("sai :", wrong_skip_start([0], 0), "| đúng:", Solution().canReach([0], 0))

sai : False | đúng: True
sai : False | đúng: True


**Vì sao sai:** khi `arr[start] == 0`, các bước nhảy đều về chính `start` (đã trong `seen`) nên không bao giờ "phát hiện" 0. **Sửa:** kiểm tra `arr[i] == 0` cho **đỉnh lấy ra** (gồm cả `start`), như bản nộp.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Từ vị trí X, mỗi bước có vài lựa chọn; có tới được Y không?"** ⇒ đồ thị ẩn + DFS/BFS + `visited`.
- **"Số bước ít nhất"** ⇒ BFS (mỗi lớp = +1 bước).
- Đỉnh là số nguyên `0..n−1` ⇒ dùng `list` boolean thay cho `set` (nhanh hơn).
- Dữ liệu **không âm** và được phép sửa input ⇒ đánh dấu bằng **đổi dấu**.
- Python + độ sâu có thể > 1000 ⇒ ưu tiên BFS / stack tường minh.
- Đừng quên: chỉ số âm trong Python **không** ném lỗi — phải tự kiểm tra `j >= 0`.

## 15. Biến thể của bài

1. **LeetCode 55 — Jump Game**: nhảy tối đa `arr[i]` bước về phía trước, có tới cuối không ⇒ tham lam `O(n)`.
2. **LeetCode 45 — Jump Game II**: số bước ít nhất tới cuối ⇒ BFS theo "khoảng".
3. **LeetCode 1345 — Jump Game IV**: thêm cạnh giữa các chỉ số cùng giá trị ⇒ BFS + gom nhóm theo giá trị.
4. **LeetCode 1340 — Jump Game V**, **1871 — Jump Game VII**: DP / BFS với cửa sổ trượt.
5. **Số bước ít nhất để tới một số 0** trong Jump Game III (bài tập 5).
6. **Liệt kê mọi chỉ số 0 đến được** — chính là tập `seen` sau khi duyệt xong.

In [28]:
# Biến thể 6: liệt kê các chỉ số mang giá trị 0 mà đến được
def reachable_zeros(arr, start):
    n = len(arr)
    seen = [False] * n
    seen[start] = True
    q = deque([start])
    while q:
        i = q.popleft()
        for j in (i + arr[i], i - arr[i]):
            if 0 <= j < n and not seen[j]:
                seen[j] = True
                q.append(j)
    return [i for i in range(n) if seen[i] and arr[i] == 0]

print(reachable_zeros([0, 3, 0, 6, 3, 3, 4], 6))
print(reachable_zeros([3, 0, 2, 1, 2], 2))

[2]
[]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Lan truyền lặp | Quét cả mảng đến khi ổn định | `O(n²)` | `O(n)` | Minh hoạ, kiểm chứng |
| 2. DFS đệ quy | `dfs(i)` + `seen`, dừng sớm bằng `or` | `O(n)` | `O(n)` | Code ngắn, cần nâng limit |
| 3. BFS | Hàng đợi + đánh dấu khi đẩy | `O(n)` | `O(n)` | Bản nộp chuẩn; mở rộng ra "ít bước nhất" |
| 4. DFS lặp, đổi dấu | Không cần `seen` | `O(n)` | `O(n)` stack | Khi được sửa input, muốn tiết kiệm bộ nhớ |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().canReach([0], 0)`, `Solution().canReach([1, 1, 1], 1)`, `Solution().canReach([2, 0, 2], 0)`?

**Bài 2 (tính tay):** Ghi thứ tự các chỉ số được **lấy ra** khi chạy BFS (cách 3) với `arr = [4,2,3,0,3,1,2]`, `start = 0`.

**Bài 3 (điền TODO):** Hoàn thành `valid_jumps(arr, i)` trả list các đích nhảy hợp lệ từ `i` (không trùng lặp khi `arr[i] == 0`).

**Bài 4 (tự viết lại):** Viết lại DFS bằng stack tường minh + mảng `seen` (không đổi dấu): `my_can_reach(arr, start)`.

**Bài 5 (mini challenge):** Viết `min_jumps_to_zero(arr, start)` trả **số bước nhảy ít nhất** để tới một chỉ số có giá trị 0, hoặc `-1` nếu không thể.

In [29]:
# Bài 3, 4, 5 — khung để tự điền (KHÔNG gọi khi chạy notebook)
def valid_jumps_todo(arr, i):
    # TODO: xét i + arr[i] và i - arr[i], giữ trong biên, bỏ trùng
    pass

def my_can_reach_todo(arr, start):
    # TODO: stack + seen
    pass

def min_jumps_to_zero_todo(arr, start):
    # TODO: BFS theo lớp, lưu khoảng cách
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) `[0]`: `start` đã là 0. (2) `[1,1,1]` không có số 0 nào. (3) `[2,0,2]` từ 0 nhảy tới 2, từ 2 nhảy về 0 — chỉ số 1 không bao giờ tới được.

**Bài 2:** (1) Từ 0 chỉ nhảy được tới 4. (2) Từ 4: `7` ngoài biên, `1` hợp lệ. (3) Từ 1: tới 3 (giá trị 0).

**Bài 3:** (1) Hai ứng viên `i ± arr[i]`. (2) Lọc `0 <= j < n`. (3) Khi `arr[i] == 0` hai ứng viên trùng nhau ⇒ dùng `set` hoặc kiểm tra.

**Bài 4:** (1) `stack = [start]`, `seen[start] = True`. (2) `pop`, kiểm tra 0. (3) Đẩy hàng xóm chưa thăm và đánh dấu ngay.

**Bài 5:** (1) BFS với `dist[start] = 0`. (2) `dist[j] = dist[i] + 1` khi đẩy `j`. (3) Đỉnh đầu tiên lấy ra có `arr[i] == 0` cho khoảng cách nhỏ nhất (BFS duyệt theo khoảng cách tăng dần).

## 19. Đáp án tham khảo cho bài tập

In [30]:
# Bài 1
assert Solution().canReach([0], 0) is True
assert Solution().canReach([1, 1, 1], 1) is False
assert Solution().canReach([2, 0, 2], 0) is False

# Bài 2
def bfs_pop_order(arr, start):
    n = len(arr); seen = [False] * n; seen[start] = True
    q = deque([start]); order = []
    while q:
        i = q.popleft(); order.append(i)
        if arr[i] == 0:
            break
        for j in (i + arr[i], i - arr[i]):
            if 0 <= j < n and not seen[j]:
                seen[j] = True; q.append(j)
    return order
assert bfs_pop_order([4, 2, 3, 0, 3, 1, 2], 0) == [0, 4, 1, 3]

# Bài 3
def valid_jumps(arr, i):
    n = len(arr)
    res = []
    for j in (i + arr[i], i - arr[i]):
        if 0 <= j < n and j not in res:
            res.append(j)
    return res
assert valid_jumps([4, 2, 3, 0, 3, 1, 2], 5) == [6, 4]
assert valid_jumps([4, 2, 3, 0, 3, 1, 2], 3) == [3]
assert valid_jumps([4, 2, 3, 0, 3, 1, 2], 0) == [4]
print("Bài 1–3: OK")

Bài 1–3: OK


In [31]:
# Bài 4
def my_can_reach(arr, start):
    n = len(arr)
    seen = [False] * n
    seen[start] = True
    stack = [start]
    while stack:
        i = stack.pop()
        if arr[i] == 0:
            return True
        for j in (i + arr[i], i - arr[i]):
            if 0 <= j < n and not seen[j]:
                seen[j] = True
                stack.append(j)
    return False

# Bài 5
def min_jumps_to_zero(arr, start):
    n = len(arr)
    dist = [-1] * n
    dist[start] = 0
    q = deque([start])
    while q:
        i = q.popleft()
        if arr[i] == 0:
            return dist[i]
        for j in (i + arr[i], i - arr[i]):
            if 0 <= j < n and dist[j] == -1:
                dist[j] = dist[i] + 1
                q.append(j)
    return -1

def min_jumps_brute(arr, start):
    # lan truyền theo lớp: tập đỉnh đến được sau đúng k bước (k <= n)
    n = len(arr)
    frontier, seen = {start}, {start}
    for k in range(n + 1):
        if any(arr[i] == 0 for i in frontier):
            return k
        nxt = set()
        for i in frontier:
            for j in (i + arr[i], i - arr[i]):
                if 0 <= j < n and j not in seen:
                    nxt.add(j)
        seen |= nxt
        frontier = nxt
        if not frontier:
            return -1
    return -1

assert min_jumps_to_zero([4, 2, 3, 0, 3, 1, 2], 5) == 3
assert min_jumps_to_zero([3, 0, 2, 1, 2], 2) == -1
assert min_jumps_to_zero([0], 0) == 0
rng = random.Random(61)
for _ in range(1500):
    n = rng.randint(1, 20)
    arr = [rng.randint(0, n - 1) for _ in range(n)]
    s = rng.randrange(n)
    assert my_can_reach(arr, s) == can_reach_brute(arr, s)
    assert min_jumps_to_zero(arr, s) == min_jumps_brute(arr, s)
    assert (min_jumps_to_zero(arr, s) != -1) == can_reach_brute(arr, s)
print("Bài 4–5: OK — mọi assert đều qua")

Bài 4–5: OK — mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template BFS reachability trên đồ thị ẩn:**

```python
seen = [False] * n
seen[start] = True
q = deque([start])
while q:
    i = q.popleft()
    if is_target(i):
        return True
    for j in next_states(i):
        if 0 <= j < n and not seen[j]:
            seen[j] = True          # đánh dấu khi ĐẨY
            q.append(j)
return False
```

**Checklist trước khi submit:**
- [ ] Có `visited` (hoặc đánh dấu tại chỗ bằng giá trị **không nhầm** với dữ liệu).
- [ ] Biên: `0 <= j < n` (nhớ chỉ số âm trong Python không báo lỗi).
- [ ] Kiểm tra đích cho **mọi** đỉnh lấy ra, kể cả `start`.
- [ ] Không đệ quy sâu `5·10^4` mà không nâng limit — tốt nhất dùng BFS.
- [ ] Đúng chữ ký: `canReach(self, arr: list[int], start: int) -> bool`.